# Notebook 3 — Grad-CAM Explainability
**RSNA Intracranial Hemorrhage Detection — Improved Pipeline**

Visual explanations for the 2.5D EfficientNet-B4 model using:
- **Grad-CAM** heatmaps for all 4 outcome categories (TP, FP, FN, TN)
- **Per-subtype** activation maps (which areas drive each subtype prediction)
- **Occlusion sensitivity** maps with **quantitative Δ-probability** measurement

> **⚠️ Honest Disclosure — Grad-CAM Limitations**:
> - Grad-CAM alignment with true pathology is assessed **qualitatively** through
>   visual inspection. No ground-truth segmentation masks are available in the
>   RSNA ICH dataset, so **no quantitative IoU/Dice overlap** can be measured.
> - Occlusion sensitivity provides a **quantitative** proxy: we report mean Δ probability
>   drop when the Grad-CAM-highlighted region vs. a random region is occluded.
>   This is indirect evidence but not a substitute for radiologist annotation.

### Requirements
- **GPU**: Yes (inference only)
- **Inputs**: NB01 best model (any completed fold), NB00 manifest, NB02 cache
- **Time**: ~15–30 min

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# ██  CONFIG + IMPORTS  ██
# ══════════════════════════════════════════════════════════════════════════
from pathlib import Path
import os, json, random, warnings, gc
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')

import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score, roc_curve
import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from tqdm.auto import tqdm

# ─── Paths ────────────────────────────────────────────────────────────
NB00_DIR = Path('/kaggle/input/notebooks/harshitghosh/00-preprocess-metadata')
NB02_DIR = Path('/kaggle/input/notebooks/harshitghosh/nb02eda')
NB01_DIR = Path('/kaggle/input/notebooks/harshitghosh/01-training')  # final session

MANIFEST_PATH  = NB00_DIR / 'manifest_2_5d.csv'
NPY_CACHE_DIR  = NB02_DIR / 'cache'
NORM_STATS     = NB02_DIR / 'normalization_stats.json'

FOLD_ID   = 0     # Which fold's model to visualize
MODEL_PATH = NB01_DIR / f'best_model_fold{FOLD_ID}.pth'

IMG_SIZE  = 256
BACKBONE  = 'tf_efficientnet_b4'
IN_CH     = 9
N_CLASSES = 6
DROPOUT   = 0.3
N_SAMPLES_PER_CATEGORY = 5  # TP, FP, FN, TN each

SUBTYPES = ['any', 'epidural', 'intraparenchymal',
            'intraventricular', 'subarachnoid', 'subdural']

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
OUTPUT_DIR = Path('/kaggle/working')

# Norm stats
if NORM_STATS.exists():
    _ns = json.load(open(NORM_STATS))
    MEAN_3CH = np.array(_ns['mean'], dtype=np.float32)
    STD_3CH  = np.array(_ns['std'],  dtype=np.float32)
else:
    MEAN_3CH = np.array([0.485, 0.456, 0.406], dtype=np.float32)
    STD_3CH  = np.array([0.229, 0.224, 0.225], dtype=np.float32)
MEAN_9CH = np.tile(MEAN_3CH, 3)
STD_9CH  = np.tile(STD_3CH, 3)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print(f'Device: {DEVICE}')
print(f'Model : {MODEL_PATH}')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# ██  LOAD MODEL + DATA  ██
# ══════════════════════════════════════════════════════════════════════════

def build_model(backbone=BACKBONE, in_ch=IN_CH, n_cls=N_CLASSES):
    model = timm.create_model(backbone, pretrained=False,
                              num_classes=0, drop_rate=DROPOUT, drop_path_rate=0.2)
    old = model.conv_stem
    new = nn.Conv2d(in_ch, old.out_channels, kernel_size=old.kernel_size,
                    stride=old.stride, padding=old.padding, bias=(old.bias is not None))
    model.conv_stem = new
    model.classifier = nn.Sequential(nn.Dropout(DROPOUT), nn.Linear(model.num_features, n_cls))
    return model


model = build_model()
state = torch.load(str(MODEL_PATH), map_location=DEVICE)
model.load_state_dict(state)
model = model.to(DEVICE).eval()
print(f'Model loaded: {sum(p.numel() for p in model.parameters())/1e6:.1f}M params')

# Dataset
manifest = pd.read_csv(MANIFEST_PATH)
val_df   = manifest[manifest['fold'] == FOLD_ID].reset_index(drop=True)
print(f'Validation set: {len(val_df):,} images')

val_transform = A.Compose([
    A.Normalize(mean=MEAN_9CH.tolist(), std=STD_9CH.tolist(), max_pixel_value=255),
    ToTensorV2()])

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# ██  INFERENCE: compute predictions + find TP/FP/FN/TN  ██
# ══════════════════════════════════════════════════════════════════════════

def load_2_5d(row):
    """Load a 2.5D image from manifest row."""
    def _load(iid):
        if pd.isna(iid): return np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
        p = NPY_CACHE_DIR / f'{iid}.npy'
        return np.load(str(p)).astype(np.uint8) if p.exists() else \
               np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
    prev = _load(row.get('prev_image_id'))
    center = _load(row['image_id'])
    nxt = _load(row.get('next_image_id'))
    return np.concatenate([prev, center, nxt], axis=2)  # [H, W, 9]


# Forward pass on all val samples
all_probs = []
all_labels = []

for i in tqdm(range(len(val_df)), desc='Inference'):
    row = val_df.iloc[i]
    img = load_2_5d(row)
    t = val_transform(image=img)['image'].unsqueeze(0).to(DEVICE)
    with torch.no_grad(), torch.amp.autocast('cuda'):
        logits = model(t)
    all_probs.append(torch.sigmoid(logits).cpu().numpy()[0])
    all_labels.append([row[c] for c in SUBTYPES])

probs_np  = np.array(all_probs)
labels_np = np.array(all_labels)

# Optimal threshold for 'any'
fpr, tpr, thresholds = roc_curve(labels_np[:, 0], probs_np[:, 0])
best_thresh = thresholds[np.argmax(tpr - fpr)]
preds_any = (probs_np[:, 0] >= best_thresh).astype(int)
gt_any    = labels_np[:, 0].astype(int)

# Categories
tp_idx = np.where((preds_any == 1) & (gt_any == 1))[0]
fp_idx = np.where((preds_any == 1) & (gt_any == 0))[0]
fn_idx = np.where((preds_any == 0) & (gt_any == 1))[0]
tn_idx = np.where((preds_any == 0) & (gt_any == 0))[0]

print(f'\nThreshold: {best_thresh:.4f}')
print(f'TP: {len(tp_idx):,}  FP: {len(fp_idx):,}  FN: {len(fn_idx):,}  TN: {len(tn_idx):,}')

# Pick N_SAMPLES from each for visualization
cat_samples = {
    'TP': np.random.choice(tp_idx, min(N_SAMPLES_PER_CATEGORY, len(tp_idx)), replace=False),
    'FP': np.random.choice(fp_idx, min(N_SAMPLES_PER_CATEGORY, len(fp_idx)), replace=False),
    'FN': np.random.choice(fn_idx, min(N_SAMPLES_PER_CATEGORY, len(fn_idx)), replace=False),
    'TN': np.random.choice(tn_idx, min(N_SAMPLES_PER_CATEGORY, len(tn_idx)), replace=False),
}

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# ██  GRAD-CAM IMPLEMENTATION  ██
# ══════════════════════════════════════════════════════════════════════════

class GradCAM:
    """Grad-CAM for EfficientNet (hooks on the last conv block)."""
    def __init__(self, model, target_layer=None):
        self.model = model
        # Default: last block in EfficientNet
        self.target_layer = target_layer or model.blocks[-1]
        self.activations = None
        self.gradients = None

        self.target_layer.register_forward_hook(self._fwd_hook)
        self.target_layer.register_full_backward_hook(self._bwd_hook)

    def _fwd_hook(self, module, inp, out):
        self.activations = out.detach()

    def _bwd_hook(self, module, grad_in, grad_out):
        self.gradients = grad_out[0].detach()

    def __call__(self, input_tensor, class_idx=0):
        """Returns heatmap [H, W] in [0, 1]."""
        self.model.eval()
        self.model.zero_grad()

        input_tensor = input_tensor.requires_grad_(True)
        logits = self.model(input_tensor)
        logits[0, class_idx].backward()

        weights = self.gradients.mean(dim=[2, 3], keepdim=True)
        cam = (weights * self.activations).sum(dim=1, keepdim=True)
        cam = F.relu(cam)
        cam = F.interpolate(cam, size=(input_tensor.shape[2], input_tensor.shape[3]),
                           mode='bilinear', align_corners=False)
        cam = cam[0, 0].cpu().numpy()
        if cam.max() > 0:
            cam = cam / cam.max()
        return cam


gradcam = GradCAM(model)

# ─── SANITY CHECK: random-weights model should produce diffuse heatmaps ──
print('Running sanity check: Grad-CAM with random (untrained) weights...')
rand_model = build_model()  # random weights, no pretrained checkpoint loaded
rand_model = rand_model.to(DEVICE).eval()
gradcam_rand = GradCAM(rand_model)

# Pick a TP sample for comparison
sample_row = val_df.iloc[tp_idx[0]] if len(tp_idx) > 0 else val_df.iloc[0]
sample_img = load_2_5d(sample_row)
sample_t = val_transform(image=sample_img)['image'].unsqueeze(0).to(DEVICE)

cam_trained = gradcam(sample_t, class_idx=0)
cam_random  = gradcam_rand(sample_t, class_idx=0)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
center_img = sample_t[0, 3:6].cpu().numpy().transpose(1, 2, 0)
center_img = center_img * STD_3CH + MEAN_3CH
center_img = np.clip(center_img * 255, 0, 255).astype(np.uint8)

axes[0].imshow(center_img); axes[0].set_title('Original'); axes[0].axis('off')
axes[1].imshow(center_img); axes[1].imshow(cam_trained, alpha=0.5, cmap='jet')
axes[1].set_title('Trained model'); axes[1].axis('off')
axes[2].imshow(center_img); axes[2].imshow(cam_random, alpha=0.5, cmap='jet')
axes[2].set_title('Random weights (sanity check)'); axes[2].axis('off')

plt.suptitle('Grad-CAM Sanity Check: Trained vs Random Weights', fontsize=13)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'gradcam_sanity_check.png', dpi=150, bbox_inches='tight')
plt.show()

# Quantify: trained model should have more focused activations
spatial_entropy_trained = -(cam_trained[cam_trained > 0] * np.log(cam_trained[cam_trained > 0] + 1e-9)).sum()
spatial_entropy_random  = -(cam_random[cam_random > 0] * np.log(cam_random[cam_random > 0] + 1e-9)).sum()
print(f'  Spatial entropy — trained: {spatial_entropy_trained:.2f}, random: {spatial_entropy_random:.2f}')
print(f'  {"✅" if spatial_entropy_trained < spatial_entropy_random else "⚠️"} '
      f'Trained model heatmap is {"more" if spatial_entropy_trained < spatial_entropy_random else "NOT more"} '
      f'focused than random (expected: trained < random)')

del rand_model, gradcam_rand; torch.cuda.empty_cache(); gc.collect()
print('\nGrad-CAM ready.')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# ██  VISUALIZE: TP / FP / FN / TN  ██
# ══════════════════════════════════════════════════════════════════════════

def denorm_center_slice(img_9ch_tensor):
    """Extract center slice (channels 3:6) and denormalize for display."""
    center = img_9ch_tensor[0, 3:6].cpu().numpy().transpose(1, 2, 0)
    center = center * STD_3CH + MEAN_3CH
    center = np.clip(center * 255, 0, 255).astype(np.uint8)
    return center


fig, axes = plt.subplots(4, N_SAMPLES_PER_CATEGORY, figsize=(3*N_SAMPLES_PER_CATEGORY, 12))

for row_i, (cat_name, indices) in enumerate(cat_samples.items()):
    for col_i, df_idx in enumerate(indices):
        row = val_df.iloc[df_idx]
        img_raw = load_2_5d(row)
        t = val_transform(image=img_raw)['image'].unsqueeze(0).to(DEVICE)

        # Grad-CAM for 'any' class (index 0)
        heatmap = gradcam(t, class_idx=0)
        center_img = denorm_center_slice(t)

        ax = axes[row_i, col_i]
        ax.imshow(center_img, cmap='gray' if center_img.shape[2] == 1 else None)
        ax.imshow(heatmap, alpha=0.4, cmap='jet')
        prob = probs_np[df_idx, 0]
        ax.set_title(f'{cat_name}  p={prob:.2f}', fontsize=9,
                     color='green' if cat_name in ('TP','TN') else 'red')
        ax.axis('off')

plt.suptitle('Grad-CAM: "any hemorrhage" class', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'gradcam_tp_fp_fn_tn.png', dpi=150, bbox_inches='tight')
plt.show()
print('TP/FP/FN/TN Grad-CAM grid saved.')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# ██  PER-SUBTYPE GRAD-CAM  ██
# ══════════════════════════════════════════════════════════════════════════

# Pick a positive sample for each subtype
fig, axes = plt.subplots(len(SUBTYPES), 3, figsize=(9, 3*len(SUBTYPES)))

for sub_i, sub_name in enumerate(SUBTYPES):
    pos_mask = val_df[sub_name] == 1
    if pos_mask.sum() == 0:
        for j in range(3): axes[sub_i, j].text(0.5, 0.5, 'No +ve', ha='center'); axes[sub_i, j].axis('off')
        continue
    pos_indices = val_df[pos_mask].index.tolist()
    sample_idx = random.choice(pos_indices[:50])
    row = val_df.iloc[sample_idx]
    img_raw = load_2_5d(row)
    t = val_transform(image=img_raw)['image'].unsqueeze(0).to(DEVICE)
    center_img = denorm_center_slice(t)

    # Original
    axes[sub_i, 0].imshow(center_img)
    axes[sub_i, 0].set_title(f'{sub_name} (original)', fontsize=9)
    axes[sub_i, 0].axis('off')

    # Grad-CAM for this subtype
    heatmap = gradcam(t, class_idx=sub_i)
    axes[sub_i, 1].imshow(center_img)
    axes[sub_i, 1].imshow(heatmap, alpha=0.5, cmap='jet')
    prob = probs_np[sample_idx, sub_i]
    axes[sub_i, 1].set_title(f'Grad-CAM  p={prob:.3f}', fontsize=9)
    axes[sub_i, 1].axis('off')

    # Heatmap alone
    axes[sub_i, 2].imshow(heatmap, cmap='jet')
    axes[sub_i, 2].set_title('Heatmap', fontsize=9)
    axes[sub_i, 2].axis('off')

plt.suptitle('Per-Subtype Grad-CAM', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'gradcam_per_subtype.png', dpi=150, bbox_inches='tight')
plt.show()
print('Per-subtype Grad-CAM saved.')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# ██  OCCLUSION SENSITIVITY  ██
# ══════════════════════════════════════════════════════════════════════════

def occlusion_sensitivity(model, input_tensor, class_idx=0,
                          patch_size=32, stride=16):
    """Slide a gray patch and measure probability drop."""
    model.eval()
    B, C, H, W = input_tensor.shape
    with torch.no_grad(), torch.amp.autocast('cuda'):
        base_prob = torch.sigmoid(model(input_tensor))[0, class_idx].item()

    sensitivity = np.zeros((H, W))
    count = np.zeros((H, W))

    for y in range(0, H - patch_size + 1, stride):
        for x in range(0, W - patch_size + 1, stride):
            occluded = input_tensor.clone()
            occluded[:, :, y:y+patch_size, x:x+patch_size] = 0
            with torch.no_grad(), torch.amp.autocast('cuda'):
                occ_prob = torch.sigmoid(model(occluded))[0, class_idx].item()
            drop = base_prob - occ_prob
            sensitivity[y:y+patch_size, x:x+patch_size] += drop
            count[y:y+patch_size, x:x+patch_size] += 1

    count = np.maximum(count, 1)
    return sensitivity / count


# Show for 3 TP samples
n_occ = min(3, len(tp_idx))
fig, axes = plt.subplots(n_occ, 3, figsize=(10, 3*n_occ))
if n_occ == 1: axes = axes[np.newaxis, :]

occ_stats = []  # collect quantitative Δ probability stats

for i, df_idx in enumerate(tp_idx[:n_occ]):
    row = val_df.iloc[df_idx]
    img_raw = load_2_5d(row)
    t = val_transform(image=img_raw)['image'].unsqueeze(0).to(DEVICE)
    center_img = denorm_center_slice(t)

    # Occlusion
    sens_map = occlusion_sensitivity(model, t, class_idx=0)

    # ─── Quantitative Δ probability: Grad-CAM region vs random region ───
    with torch.no_grad(), torch.amp.autocast('cuda'):
        base_prob = torch.sigmoid(model(t))[0, 0].item()

    # Grad-CAM heatmap for this sample
    heatmap_q = gradcam(t, class_idx=0)
    # Top 20% of Grad-CAM activation = "highlighted region"
    thresh_cam = np.percentile(heatmap_q, 80)
    cam_mask = heatmap_q >= thresh_cam

    # Occlude Grad-CAM region
    t_cam_occ = t.clone()
    t_cam_occ[:, :, cam_mask] = 0
    with torch.no_grad(), torch.amp.autocast('cuda'):
        cam_occ_prob = torch.sigmoid(model(t_cam_occ))[0, 0].item()
    delta_cam = base_prob - cam_occ_prob

    # Occlude random region of same size
    n_pixels = cam_mask.sum()
    rand_mask = np.zeros_like(cam_mask, dtype=bool)
    rand_indices = np.random.choice(cam_mask.size, size=int(n_pixels), replace=False)
    rand_mask.ravel()[rand_indices] = True
    t_rand_occ = t.clone()
    t_rand_occ[:, :, rand_mask] = 0
    with torch.no_grad(), torch.amp.autocast('cuda'):
        rand_occ_prob = torch.sigmoid(model(t_rand_occ))[0, 0].item()
    delta_rand = base_prob - rand_occ_prob

    occ_stats.append({
        'sample': df_idx, 'base_prob': base_prob,
        'delta_cam_region': delta_cam, 'delta_random_region': delta_rand,
        'ratio': delta_cam / (delta_rand + 1e-9)
    })

    axes[i, 0].imshow(center_img); axes[i, 0].set_title('Original'); axes[i, 0].axis('off')
    axes[i, 1].imshow(sens_map, cmap='hot'); axes[i, 1].set_title('Occlusion sensitivity')
    axes[i, 1].axis('off')
    axes[i, 2].imshow(center_img); axes[i, 2].imshow(sens_map, alpha=0.5, cmap='hot')
    axes[i, 2].set_title(f'Overlay (Δp_cam={delta_cam:.3f})'); axes[i, 2].axis('off')

plt.suptitle('Occlusion Sensitivity (TP samples)', fontsize=13)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'occlusion_sensitivity.png', dpi=150, bbox_inches='tight')
plt.show()

# ─── Print quantitative summary ───
print('\n' + '='*65)
print('  QUANTITATIVE OCCLUSION: Grad-CAM region vs Random region')
print('='*65)
print(f'{"Sample":>8s}  {"Base p":>8s}  {"Δp(CAM)":>10s}  {"Δp(rand)":>10s}  {"Ratio":>8s}')
print('-' * 55)
for s in occ_stats:
    print(f'{s["sample"]:>8d}  {s["base_prob"]:>8.4f}  {s["delta_cam_region"]:>+10.4f}  '
          f'{s["delta_random_region"]:>+10.4f}  {s["ratio"]:>8.2f}x')

mean_delta_cam  = np.mean([s['delta_cam_region']  for s in occ_stats])
mean_delta_rand = np.mean([s['delta_random_region'] for s in occ_stats])
print(f'\n  Mean Δp(Grad-CAM region): {mean_delta_cam:+.4f}')
print(f'  Mean Δp(random region):   {mean_delta_rand:+.4f}')
print(f'  Ratio: {mean_delta_cam/(mean_delta_rand+1e-9):.2f}x')
print(f'\n  Interpretation: Occluding the Grad-CAM-highlighted region causes')
print(f'  ~{mean_delta_cam/(mean_delta_rand+1e-9):.1f}x larger probability drop than a same-size random')
print(f'  region, providing quantitative evidence that Grad-CAM localizes')
print(f'  decision-relevant features. However, without ground-truth masks,')
print(f'  we cannot confirm these regions correspond to true pathology.')
print('Occlusion sensitivity saved.')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# ██  HEALTH CHECK  ██
# ══════════════════════════════════════════════════════════════════════════
errors = []
expected_files = ['gradcam_tp_fp_fn_tn.png', 'gradcam_per_subtype.png',
                  'occlusion_sensitivity.png', 'gradcam_sanity_check.png']
for f in expected_files:
    if not (OUTPUT_DIR / f).exists():
        errors.append(f'{f} missing')

health = {
    'notebook': '03_gradcam', 'status': 'PASS' if not errors else 'FAIL',
    'errors': errors,
    'tp_count': len(tp_idx), 'fp_count': len(fp_idx),
    'fn_count': len(fn_idx), 'tn_count': len(tn_idx),
    'threshold': float(best_thresh),
    'sanity_check_entropy_trained': float(spatial_entropy_trained),
    'sanity_check_entropy_random': float(spatial_entropy_random),
    'occlusion_mean_delta_cam': float(mean_delta_cam),
    'occlusion_mean_delta_rand': float(mean_delta_rand),
}
json.dump(health, open(OUTPUT_DIR / 'health_check_nb03.json', 'w'), indent=2)

if errors:
    print('❌ HEALTH CHECK FAILED:', errors)
else:
    print('✅ HEALTH CHECK PASSED — all plots saved.')